In [2]:
# API-key setup — DO NOT hard-code your key in this cell.
import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.environ["GROQ_API_KEY"]

from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


In [5]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
# def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
#             temperature=0.7, max_tokens=500):
#     response = client.chat.completions.create(
#         model=MODEL,
#         messages=[
#             {"role": "system", "content": system_prompt},
#             {"role": "user",   "content": user_prompt},
#         ],
#         temperature=temperature,
#         max_tokens=max_tokens,
#     )
#     return response.choices[0].message.content
#
# TODO: Call it once with a simple question and print the answer.
# TODO: Print response.usage as well — how many tokens did your call consume?

def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content, response.usage

answer, usage = ask_llm("What is the name of the president of Cameroon?")
print(answer)
print(usage)

The President of Cameroon is Paul Biya. He has been serving as the President of Cameroon since November 6, 1982.
CompletionUsage(completion_tokens=28, prompt_tokens=51, total_tokens=79, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.041307878, prompt_time=0.00243541, completion_time=0.065505454, total_time=0.067940864)


# Part 1.1 : Anatomy of a call

## 1. System vs user role: 
The system prompt is where you set up how the model should behave for the whole conversation, its role, tone, or any constraints you want it to follow. It's instructions for the model itself, not really something the model is "replying" to. For example, in this lab a good system prompt might be something like "You are a financial analyst assistant that only summarizes facts stated in the application, and never assumes anything the applicant didn't say." That sets the ground rules before any real question comes in.

The user role is the actual input or question you're asking the model to respond to, like the loan application text itself, or "Summarize this application in three sentences." It changes every time you call the model, while the system prompt usually stays the same across many calls.

So basically: system = the personality and rules, user = the specific request.

## 2. What is a token

A token is roughly a chunk of text, sometimes a whole word, sometimes part of a word, sometimes just punctuation. For example "microfinance" might get split into two tokens like "micro" and "finance" depending on how common the word is in the model's training data. Common short words like "the" or "is" are usually one token each.

Providers bill per token instead of per request because the actual cost to them scales with how much text the model has to process and generate, not with how many times you hit the API. A request asking for a one sentence answer costs way less compute than a request asking the model to read a 2,000 word loan application and generate a full page summary, even though both are technically "one request." 

In [7]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
question = "Suggest a name for a savings product for market traders in Accra."

low_temp_answers = []
for i in range(5):
    answer, _ = ask_llm(question, temperature=0.0)
    low_temp_answers.append(answer)

high_temp_answers = []
for i in range(5):
    answer, _ = ask_llm(question, temperature=1.2)
    high_temp_answers.append(answer)

# TODO: Print all 10 answers, grouped by temperature.
print("Temperature 0.0")
for i, ans in enumerate(low_temp_answers, 1):
    print(f"{i}. {ans}")

print("\nTemperature 1.2")
for i, ans in enumerate(high_temp_answers, 1):
    print(f"{i}. {ans}")

Temperature 0.0
1. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Market Fund**: This name is straightforward and clearly communicates the product's purpose.
4. **Sika Kokoo**: "Sika" means "money" in the Akan language, and "Kokoo" means "gather" or "collect". This name could appeal to market traders who speak Akan.
5. **Market Booster**: This name suggests that the savings product will help market traders boost their businesses.
6. **Kae Dzi**: "Kae Dzi" is a Ghanaian phrase that means "save for the future". This name could appeal to market traders who value saving for long-term goals.
7. **Traders' Trust**: This name emphasizes the idea of trust and reliability, which is important for a savings product.

Choose the one that resonates with your

# Part 1.2 Temperature: the randomness dial

## What did you observe at each temperature?
At temperature 0.0, I expected the five answers to be identical, but they weren't quite. Some names repeated in almost every run, like "Makola Save/Savings" and "Traders' Trust/Treasure", and runs 2 and 3 were word for word the same. But each run still had a few different names mixed in, so 0.0 was mostly consistent, not perfectly deterministic.
At temperature 1.2, the answers were clearly more varied. Each run had a mostly different set of names, some more unusual, like "SikaBox" and "Obaa Savings", that never showed up at 0.0. A few names like "Makola Savings" and "Kokroko Savings" still repeated, but overall there was much more spread than at 0.0.

## Which temperature is appropriate for the loan system?
Low temperature, close to 0.0, is the right choice. A loan officer needs the same application to produce the same summary, extracted data, and recommendation every time it's processed. Even the small variation I saw at 0.0 is something to minimize, not add to, so high temperature would only make the output less trustworthy for this use case.